# Modeling - WESAD Wrist Features

Baseline models included: Logistic Regression, Random Forest, SVM, MLP, and XGBoost (if available).

Outputs are saved to:
- `models/`
- `reports/metrics.json`
- `reports/figs/`

In [10]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import joblib

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

print('Imports loaded')

Imports loaded


In [11]:
DATA_PATH = Path('wesad_wrist_features.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing file: {DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH)
print('shape:', df.shape)
display(df.head())

shape: (179817, 46)


,subject,window_index,label,bvp_hr_mean,bvp_hr_std,bvp_hrv_mean,bvp_hrv_std,bvp_hrv_nn50,bvp_hrv_pnn50,bvp_hrv_rmssd,...,acc_mag_mean,acc_mag_std,acc_x_absint,acc_y_absint,acc_z_absint,acc_x_peakfreq,acc_y_peakfreq,acc_z_peakfreq,eda_scr_has_peaks,bvp_hrv_freq_valid
0,S10,0,1,92.222633,30.733178,729.552469,259.659106,60,0.750000,391.919729,...,61.879907,3.012126,2384.783203,202.807617,2570.923828,0.125,0.125,0.125,1,1
1,S10,1,1,92.222633,30.733178,729.552469,259.659106,60,0.750000,391.919729,...,61.877399,3.008428,2377.814453,201.671875,2578.465820,0.125,0.125,0.125,1,1
2,S10,2,1,92.743837,30.566826,723.828125,256.147262,59,0.746835,389.281709,...,61.873905,3.010429,2370.814453,200.614258,2585.978516,0.125,0.125,0.125,1,1
3,S10,3,1,92.652350,30.388575,723.572531,254.571460,59,0.737500,386.844992,...,61.870738,3.011397,2363.833008,199.660156,2593.484375,0.125,0.125,0.125,1,1
4,S10,4,1,92.629447,30.394781,723.765432,254.561813,59,0.737500,386.856825,...,61.868780,3.009604,2356.885742,198.826172,2601.025391,0.125,0.125,0.125,1,1


In [12]:
required_cols = ['subject', 'label']
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

if 'window_index' not in df.columns:
    df['window_index'] = np.arange(len(df))

df = df[df['label'].notna()].copy()

null_pct = df.isna().mean()
drop_cols = null_pct[null_pct > 0.5].index.tolist()
drop_cols = [c for c in drop_cols if c not in ['subject', 'label', 'window_index']]
df = df.drop(columns=drop_cols)

id_cols = ['subject', 'window_index', 'label']
feature_cols = [c for c in df.columns if c not in id_cols]
numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

if not numeric_cols:
    raise ValueError('No numeric feature columns found after preprocessing.')

X = df[numeric_cols].copy()
y_orig = df['label'].astype(int).to_numpy()
y = y_orig - 1  # Convert to 0-indexed for sklearn/xgboost compatibility
groups = df['subject'].to_numpy()
classes = np.sort(np.unique(y))

print('rows:', len(df))
print('subjects:', df['subject'].nunique())
print('original classes (in data):', np.sort(np.unique(y_orig)).tolist())
print('converted classes (0-indexed):', classes.tolist())
print('num features:', len(numeric_cols))

rows: 179817
subjects: 15
original classes (in data): [1, 2, 3, 4]
converted classes (0-indexed): [0, 1, 2, 3]
num features: 43


In [13]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

print('train size:', len(train_idx))
print('test size:', len(test_idx))
print('train subjects:', len(np.unique(groups[train_idx])))
print('test subjects:', len(np.unique(groups[test_idx])))

print('\nClass distribution (0-indexed):')
for cls in classes:
    train_cnt = np.sum(y_train == cls)
    test_cnt = np.sum(y_test == cls)
    print(f'  class {cls+1} (idx {cls}): train={train_cnt}, test={test_cnt}')

train size: 143689
test size: 36128
train subjects: 12
test subjects: 3

Class distribution (0-indexed):
  class 1 (idx 0): train=56357, test=14072
  class 2 (idx 1): train=31824, test=8040
  class 3 (idx 2): train=17836, test=4464
  class 4 (idx 3): train=37672, test=9552


In [14]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
], remainder='drop')

pipelines = {
    'logreg': Pipeline([
        ('pre', preprocessor),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1, random_state=42)),
    ]),
    'rf': Pipeline([
        ('pre', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=42)),
    ]),
    'svm': Pipeline([
        ('pre', preprocessor),
        ('clf', SVC(kernel='rbf', probability=True, random_state=42)),
    ]),
    'mlp': Pipeline([
        ('pre', preprocessor),
        ('clf', MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, early_stopping=True, random_state=42)),
    ]),
}

if XGBClassifier is not None:
    pipelines['xgb'] = Pipeline([
        ('pre', preprocessor),
        ('clf', XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric='mlogloss',
            random_state=42,
        )),
    ])
else:
    print('XGBoost unavailable, skipping xgb model')

print('models:', list(pipelines.keys()))

models: ['logreg', 'rf', 'svm', 'mlp', 'xgb']


In [15]:
def get_scores(pipe, X_data):
    if hasattr(pipe, 'predict_proba'):
        return pipe.predict_proba(X_data)
    if hasattr(pipe, 'decision_function'):
        raw = pipe.decision_function(X_data)
        if raw.ndim == 1:
            raw = np.vstack([-raw, raw]).T
        exp = np.exp(raw - np.max(raw, axis=1, keepdims=True))
        return exp / exp.sum(axis=1, keepdims=True)
    return None

results = {}
trained = {}

for name, pipe in pipelines.items():
    print(f'\nTraining {name}...')
    pipe.fit(X_train, y_train)
    trained[name] = pipe

    y_pred = pipe.predict(X_test)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    f1m = f1_score(y_test, y_pred, average='macro')
    report = classification_report(y_test, y_pred, output_dict=True)
    cm = confusion_matrix(y_test, y_pred, labels=classes)

    results[name] = {
        'balanced_accuracy': float(bal_acc),
        'f1_macro': float(f1m),
        'classification_report': report,
        'confusion_matrix': cm.tolist(),
    }

    print(f'{name}: balanced_accuracy={bal_acc:.4f}, f1_macro={f1m:.4f}')

summary = pd.DataFrame([
    {'model': n, 'balanced_accuracy': v['balanced_accuracy'], 'f1_macro': v['f1_macro']}
    for n, v in results.items()
]).sort_values('balanced_accuracy', ascending=False).reset_index(drop=True)

display(summary)


Training logreg...


/Users/ysedra/Documents/SUPSI/SUPSIvenv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


logreg: balanced_accuracy=0.5202, f1_macro=0.5352

Training rf...
rf: balanced_accuracy=0.5945, f1_macro=0.5486

Training svm...
svm: balanced_accuracy=0.5753, f1_macro=0.5757

Training mlp...
mlp: balanced_accuracy=0.5889, f1_macro=0.5851

Training xgb...
xgb: balanced_accuracy=0.5896, f1_macro=0.5686


,model,balanced_accuracy,f1_macro
0,rf,0.594546,0.548611
1,xgb,0.589569,0.568594
2,mlp,0.588875,0.585127
3,svm,0.575290,0.575681
4,logreg,0.520229,0.535184


In [16]:
os.makedirs('models', exist_ok=True)
os.makedirs('reports/figs', exist_ok=True)

for name, pipe in trained.items():
    p = Path('models') / f'{name}_pipeline.joblib'
    joblib.dump(pipe, p)
    print('saved', p)

with open('reports/metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

print('saved reports/metrics.json')

saved models/logreg_pipeline.joblib
saved models/rf_pipeline.joblib
saved models/svm_pipeline.joblib
saved models/mlp_pipeline.joblib
saved models/xgb_pipeline.joblib
saved reports/metrics.json


In [17]:
def plot_confusion_for_top_models(top_n=2):
    top = summary.head(top_n)['model'].tolist()
    for name in top:
        cm = np.array(results[name]['confusion_matrix'])
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=classes, yticklabels=classes)
        plt.title(f'Confusion Matrix - {name}')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.tight_layout()
        out = f'reports/figs/{name}_confusion.png'
        plt.savefig(out, dpi=140)
        plt.close()
        print('saved', out)

plot_confusion_for_top_models(top_n=2)

saved reports/figs/rf_confusion.png
saved reports/figs/xgb_confusion.png


In [18]:
def plot_roc_pr_curves(name, pipe):
    scores = get_scores(pipe, X_test)
    if scores is None:
        print('No score output for', name)
        return

    y_bin = label_binarize(y_test, classes=classes)
    if scores.shape[1] != y_bin.shape[1]:
        print('Score/class mismatch for', name)
        return

    plt.figure(figsize=(10, 4))
    for i, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], scores[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'class {cls} (AUC={roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f'ROC - {name}')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.tight_layout()
    out_roc = f'reports/figs/{name}_roc.png'
    plt.savefig(out_roc, dpi=140)
    plt.close()

    plt.figure(figsize=(10, 4))
    for i, cls in enumerate(classes):
        precision, recall, _ = precision_recall_curve(y_bin[:, i], scores[:, i])
        ap = average_precision_score(y_bin[:, i], scores[:, i])
        plt.plot(recall, precision, label=f'class {cls} (AP={ap:.2f})')
    plt.title(f'Precision-Recall - {name}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.legend(loc='lower left')
    plt.tight_layout()
    out_pr = f'reports/figs/{name}_pr.png'
    plt.savefig(out_pr, dpi=140)
    plt.close()

    print('saved', out_roc)
    print('saved', out_pr)

for n, p in trained.items():
    plot_roc_pr_curves(n, p)

saved reports/figs/logreg_roc.png
saved reports/figs/logreg_pr.png
saved reports/figs/rf_roc.png
saved reports/figs/rf_pr.png
saved reports/figs/svm_roc.png
saved reports/figs/svm_pr.png
saved reports/figs/mlp_roc.png
saved reports/figs/mlp_pr.png
saved reports/figs/xgb_roc.png
saved reports/figs/xgb_pr.png


In [19]:
def plot_feature_importance(name, pipe, top_n=25):
    est = pipe.named_steps['clf']
    importances = None

    if hasattr(est, 'feature_importances_'):
        importances = np.asarray(est.feature_importances_)
    elif hasattr(est, 'coef_'):
        coef = np.asarray(est.coef_)
        importances = np.mean(np.abs(coef), axis=0) if coef.ndim > 1 else np.abs(coef)

    if importances is None:
        print('No importances for', name)
        return

    idx = np.argsort(importances)[::-1][:top_n]
    vals = importances[idx]
    feats = [numeric_cols[i] for i in idx]

    plt.figure(figsize=(9, max(5, int(0.24 * top_n))))
    sns.barplot(x=vals, y=feats, orient='h')
    plt.title(f'Feature Importance - {name}')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    out = f'reports/figs/{name}_importance.png'
    plt.savefig(out, dpi=140)
    plt.close()
    print('saved', out)

for n, p in trained.items():
    plot_feature_importance(n, p, top_n=25)

saved reports/figs/logreg_importance.png
saved reports/figs/rf_importance.png
No importances for svm
No importances for mlp
saved reports/figs/xgb_importance.png


In [20]:
def plot_calibration(name, pipe, bins=10):
    scores = get_scores(pipe, X_test)
    if scores is None:
        print('No probabilities for', name)
        return

    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(classes):
        y_true_bin = (y_test == cls).astype(int)
        frac_pos, mean_pred = calibration_curve(y_true_bin, scores[:, i], n_bins=bins)
        plt.plot(mean_pred, frac_pos, marker='o', label=f'class {cls}')

    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f'Calibration - {name}')
    plt.xlabel('Mean predicted value')
    plt.ylabel('Fraction of positives')
    plt.legend()
    plt.tight_layout()
    out = f'reports/figs/{name}_calibration.png'
    plt.savefig(out, dpi=140)
    plt.close()
    print('saved', out)

for n, p in trained.items():
    plot_calibration(n, p, bins=10)

saved reports/figs/logreg_calibration.png
saved reports/figs/rf_calibration.png
saved reports/figs/svm_calibration.png
saved reports/figs/mlp_calibration.png
saved reports/figs/xgb_calibration.png


## Done
Notebook execution creates baseline models, metric reports, and evaluation figures.